# Tugas Kelompok Pertemuan 5
## Segmentasi Pelanggan dengan K-Means Clustering

**Mata Kuliah:** Pengantar Data Mining  

### Tujuan
Notebook ini adalah skeleton pengerjaan tugas. Lengkapi bagian **TODO** dan jangan menghapus penjelasan yang diperlukan.

> `churn_next_30d` **tidak boleh digunakan sebagai fitur clustering**. Gunakan hanya setelah cluster terbentuk untuk profiling.


## 0. Identitas Kelompok

- Kelompok: ...
- Anggota 1: ...
- Anggota 2: ...
- Anggota 3: ...
- Anggota 4: ...

## 1. Pertanyaan Analisis

**Tujuan segmentasi.** Menemukan kelompok pelanggan dengan pola pemakaian dan biaya yang berbeda, tanpa memakai label churn, agar perusahaan dapat merancang pendekatan layanan dan retensi yang berbeda untuk tiap kelompok.

**Mengapa relevan.** Satu program retensi untuk semua pelanggan kurang efisien: pelanggan berpemakaian ringan dan tagihan rendah, misalnya, tidak membutuhkan penawaran yang sama dengan pelanggan berpemakaian berat dan tagihan tinggi.

**Insight yang diharapkan.** (1) Berapa kelompok perilaku yang wajar ada di data; (2) apa profil tiap kelompok; (3) kelompok mana yang tampak lebih berisiko churn setelah clustering (analisis hilir, deskriptif); (4) tindakan apa yang masuk akal untuk tiap kelompok.

## 2. Import Library

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


ModuleNotFoundError: No module named 'sklearn'

## 3. Load Dataset

**TODO:** sesuaikan path dataset.


In [ ]:
DATA_PATH = "https://raw.githubusercontent.com/liebenlito/churn_dm/refs/heads/main/Project_Churn_Dataset/churn_customer_clean.csv"

try:
    df = pd.read_csv(DATA_PATH)
except Exception:
    # cadangan: file lokal / hasil upload ke Colab
    df = pd.read_csv("churn_customer_clean.csv")

print("Shape:", df.shape)
df.head()

## 4. Pemeriksaan Awal Data

In [ ]:
df.info()

In [ ]:
df.isna().sum().sort_values(ascending=False).head(20)

In [ ]:
print("Duplikat:", df.duplicated().sum())

## 5. Pemilihan Fitur Clustering

Dipilih **3 fitur numerik**. Kolom `customer_id`, `Unnamed: 0`, dan `churn_next_30d` tidak dipakai.
`total_charge` dan `historical_customer_value` tidak dipakai karena berkorelasi tinggi dengan `tenure_months` (sekitar 0,83), sehingga akan menghitung ganda informasi lama berlangganan.

| Fitur | Alasan |
|---|---|
| `tenure_months` | Lama berlangganan: membedakan pelanggan baru dan loyal. |
| `monthly_charge` | Nilai tagihan bulanan: proksi nilai pelanggan bagi perusahaan. |
| `avg_monthly_gb` | Intensitas pemakaian data: proksi kebiasaan pemakaian. |

Ketiganya numerik, tersedia sebelum kejadian churn, dan hampir tidak saling berkorelasi (tenure vs tagihan sekitar -0,02; tenure vs GB 0,00), sehingga masing-masing membawa informasi yang berbeda. Sebagai uji sensitivitas, Bagian 20 menambahkan `satisfaction_score`.

In [ ]:
features = [
    "tenure_months",
    "monthly_charge",
    "avg_monthly_gb",
]

X = df[features].copy()
X.head()

## 6. Data Preparation

Hasil pemeriksaan (lihat sel di bawah):
- **Missing values:** tidak ada pada fitur terpilih.
- **Tipe data:** ketiga fitur sudah numerik.
- **Nilai tidak valid:** `avg_monthly_gb = 0` pada sekitar 6,8% pelanggan. Jumlahnya sama persis dengan pelanggan tanpa layanan internet (`internet_service == "No"`), jadi ini nol yang sah, bukan data hilang. Baris ini **dipertahankan**.
- **Outlier:** tidak ada nilai dengan |z| > 3 pada ketiga fitur dan skewness kecil (maks. sekitar 0,4), sehingga `StandardScaler` dinilai cukup dan baris tidak dihapus.

In [ ]:
X.isna().sum()

In [ ]:
# Validasi: tipe data, nol sah pada GB, dan outlier ekstrem
print(X.dtypes, "\n")
zero_gb = (df["avg_monthly_gb"] == 0)
print("avg_monthly_gb == 0:", zero_gb.mean().round(4))
print("internet_service == 'No':", (df["internet_service"] == "No").mean().round(4))
print("Nol GB identik dengan 'tanpa internet':", (zero_gb == (df["internet_service"] == "No")).all())

z = (X - X.mean()) / X.std()
print("\nJumlah |z| > 3 per fitur:\n", (z.abs() > 3).sum())
print("\nSkewness:\n", X.skew().round(2))

## 7. Scaling

K-Means memakai jarak Euclidean kuadrat. Tanpa scaling, fitur dengan rentang besar mendominasi jarak: `avg_monthly_gb` berkisar 0 sampai sekitar 226 dan `monthly_charge` sampai sekitar 148, sedangkan `tenure_months` hanya sampai 70. Besarnya rentang belum tentu berarti pentingnya secara bisnis, jadi dipakai `StandardScaler` ($z=(x-\bar{x})/s$) agar ketiga fitur berkontribusi setara.

In [ ]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(
    X_scaled,
    columns=features,
    index=X.index
)

X_scaled.head()


## 8. Eksplorasi Singkat Fitur

In [ ]:
X.describe().T

In [ ]:
X.hist(figsize=(10,3), bins=25, layout=(1,3))
plt.tight_layout()
plt.show()

plt.figure(figsize=(5,4))
plt.imshow(X.corr(), cmap="coolwarm", vmin=-1, vmax=1)
plt.xticks(range(3), features, rotation=30, ha="right")
plt.yticks(range(3), features)
plt.colorbar(label="korelasi")
plt.title("Korelasi antar fitur")
plt.tight_layout()
plt.show()

# Bagian A — Menentukan Jumlah Cluster

## 9. K-Means untuk k = 2 sampai 6

In [ ]:
k_values = range(2, 7)

inertias = []
silhouette_scores = []

for k in k_values:
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    labels = model.fit_predict(X_scaled)

    inertias.append(model.inertia_)
    silhouette_scores.append(
        silhouette_score(X_scaled, labels)
    )

results = pd.DataFrame({
    "k": list(k_values),
    "inertia": inertias,
    "silhouette": silhouette_scores
})

results


## 10. Elbow Method / WCSS

In [ ]:
plt.figure(figsize=(7,4))
plt.plot(results["k"], results["inertia"], marker="o")
plt.xlabel("Jumlah Cluster (k)")
plt.ylabel("WCSS / Inertia")
plt.title("Elbow Method")
plt.grid(alpha=0.3)
plt.show()


**Interpretasi Elbow** (inertia: 3758 → 2773 → 2095 → 1800 → 1580 untuk k = 2…6):

Penurunan inertia tiap penambahan k adalah sekitar 984 (2→3), 678 (3→4), 296 (4→5), dan 220 (5→6). Penurunan melambat tajam setelah k = 4, jadi elbow yang masuk akal ada di **k = 4**, dengan k = 3 sebagai alternatif. Elbow tidak setegas pada ilustrasi di slide, sehingga keputusan tidak boleh bertumpu padanya saja.

## 11. Silhouette Score

In [ ]:
plt.figure(figsize=(7,4))
plt.plot(results["k"], results["silhouette"], marker="o")
plt.xlabel("Jumlah Cluster (k)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score")
plt.grid(alpha=0.3)
plt.show()


**Interpretasi Silhouette** (k = 2…6: 0,311; 0,317; 0,322; 0,310; 0,295):

Silhouette tertinggi ada pada **k = 4 (0,322)**, tetapi selisihnya dengan k = 2, 3, dan 5 kecil. Nilai di sekitar 0,3 menandakan struktur cluster yang **lemah sampai sedang**: kelompok memang terbedakan, namun tidak terpisah tegas dan ada tumpang tindih. Hasil ini searah dengan Elbow, yang sama-sama menunjuk k = 4.

## 12. Pilih Nilai k Final

**Nilai k yang dipilih: 4**

Alasan:
- **WCSS/Elbow:** penurunan melambat tajam setelah k = 4.
- **Silhouette:** puncak pada k = 4 (0,322), walau selisihnya tipis.
- **Interpretabilitas:** keempat cluster punya profil yang berbeda dan mudah dinamai (lihat bagian 15). Pada k = 3 kelompok pelanggan baru (tenure pendek) dan pelanggan tagihan rendah-berpemakaian ringan tidak terpisah.
- **Ukuran cluster:** 402, 681, 398, dan 513 pelanggan (sekitar 20% sampai 34%), tidak ada cluster yang terlalu kecil.
- **Stabilitas:** pengulangan dengan seed berbeda memberi ARI ≥ 0,995 terhadap model utama (lihat sel di bawah).
- **Kegunaan bisnis:** empat segmen masih realistis untuk dibuatkan empat pendekatan layanan.

# Bagian B — Model K-Means Final

In [ ]:
BEST_K = 4

assert BEST_K is not None, "Isi BEST_K terlebih dahulu."

kmeans_final = KMeans(
    n_clusters=BEST_K,
    random_state=42,
    n_init=10
)

labels = kmeans_final.fit_predict(X_scaled)

df_result = df.loc[X_scaled.index].copy()
df_result["cluster"] = labels

df_result["cluster"].value_counts().sort_index()

In [ ]:
from sklearn.metrics import adjusted_rand_score

# Stabilitas: bandingkan label model utama dengan seed lain
for s in [1, 7, 99]:
    other = KMeans(n_clusters=BEST_K, random_state=s, n_init=10).fit_predict(X_scaled)
    print(f"seed={s}: ARI vs model utama = {adjusted_rand_score(labels, other):.3f}")

## 13. Ukuran Setiap Cluster

In [ ]:
cluster_size = (
    df_result["cluster"]
    .value_counts()
    .sort_index()
    .rename("n_customers")
    .to_frame()
)

cluster_size["percentage"] = (
    cluster_size["n_customers"] / len(df_result) * 100
)

cluster_size


## 14. Profiling Cluster

In [ ]:
cluster_profile_mean = (
    df_result
    .groupby("cluster")[features]
    .mean()
    .round(2)
)

cluster_profile_mean


In [ ]:
cluster_profile_median = (
    df_result
    .groupby("cluster")[features]
    .median()
    .round(2)
)

cluster_profile_median


## 15. Nama / Persona Cluster

Nomor cluster dapat berbeda jika notebook dijalankan ulang di lingkungan lain. Cocokkan nama dengan tabel profil pada bagian 14, bukan dengan nomornya. Berikut hasil pada `random_state=42`:

| Cluster | Nama / Persona | Karakteristik Utama (median) |
|---|---|---|
| 0 | **Pemakai Ringan Hemat** | Tagihan paling rendah (sekitar 57), pemakaian data sangat kecil (sekitar 28 GB), tenure sedang (27 bulan). |
| 1 | **Pelanggan Baru Menengah** | Tenure paling pendek (19 bulan), tagihan menengah (sekitar 82), pemakaian sedang (sekitar 94 GB); jumlah terbesar (681). |
| 2 | **Pemakai Berat Tagihan Tinggi** | Tagihan tertinggi (sekitar 115), pemakaian data paling tinggi (sekitar 159 GB), tenure sedang (28 bulan). |
| 3 | **Loyal Jangka Panjang** | Tenure paling panjang (47 bulan), tagihan menengah (sekitar 79), pemakaian sedang (sekitar 89 GB); porsi kontrak dua tahun paling besar (sekitar 30%). |

## 16. Visualisasi Hasil Clustering

In [ ]:
# TODO: sesuaikan dua fitur yang akan diplot
x_feature = features[0]
y_feature = features[1]

plt.figure(figsize=(8,5))

for c in sorted(df_result["cluster"].unique()):
    subset = df_result[df_result["cluster"] == c]
    plt.scatter(
        subset[x_feature],
        subset[y_feature],
        label=f"Cluster {c}",
        alpha=0.65
    )

plt.xlabel(x_feature)
plt.ylabel(y_feature)
plt.title("Visualisasi Hasil Clustering")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


In [ ]:
# Visualisasi kedua: profil cluster (z-score rata-rata fitur) dan proyeksi PCA 2D
from sklearn.decomposition import PCA

fig, axes = plt.subplots(1, 2, figsize=(13,4.5))

prof_z = X_scaled.assign(cluster=labels).groupby("cluster")[features].mean()
prof_z.plot(kind="bar", ax=axes[0])
axes[0].axhline(0, color="black", lw=0.8)
axes[0].set_ylabel("Rata-rata z-score")
axes[0].set_title("Profil cluster (relatif terhadap rata-rata populasi)")
axes[0].tick_params(axis="x", rotation=0)
axes[0].grid(axis="y", alpha=0.3)

pca = PCA(n_components=2, random_state=42)
pcs = pca.fit_transform(X_scaled)
for c in sorted(df_result["cluster"].unique()):
    m = labels == c
    axes[1].scatter(pcs[m,0], pcs[m,1], s=12, alpha=0.6, label=f"Cluster {c}")
axes[1].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.0f}% varians)")
axes[1].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.0f}% varians)")
axes[1].set_title("Proyeksi PCA 2D")
axes[1].legend()
axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.show()

# Bagian C — Profiling Churn Setelah Clustering

## 17. Analisis `churn_next_30d`

Target dipakai hanya **setelah** clustering. Churn rate keseluruhan sekitar 16,8%.

Hasil pada `random_state=42` (periksa ulang dengan output sel di bawah):

| Cluster | Churn rate |
|---|---|
| 0 Pemakai Ringan Hemat | sekitar 11,4% |
| 1 Pelanggan Baru Menengah | sekitar 20,0% |
| 2 Pemakai Berat Tagihan Tinggi | sekitar 23,4% |
| 3 Loyal Jangka Panjang | sekitar 11,5% |

1. **Churn tertinggi:** cluster 2 (Pemakai Berat Tagihan Tinggi), disusul cluster 1 (Pelanggan Baru Menengah).
2. **Karakteristik cluster tersebut:** tagihan paling tinggi dan pemakaian data paling besar; pelanggan bernilai tinggi yang paling berisiko.
3. **Sebab-akibat?** Tidak. Ini analisis deskriptif asosiasi: cluster dibentuk dari tenure, tagihan, dan pemakaian, lalu churn dibandingkan sesudahnya. Tidak ada kontrol terhadap faktor lain (kepuasan, kualitas jaringan, promo pesaing), dan perbedaan antar-cluster belum diuji secara statistik. Untuk mengetahui penyebab atau memprediksi churn diperlukan model klasifikasi atau eksperimen.

In [ ]:
TARGET = "churn_next_30d"

if TARGET in df_result.columns:
    # Kolom target bertipe teks ("Yes"/"No"), jadi diubah ke 1/0 terlebih dahulu
    churn_flag = (df_result[TARGET] == "Yes").astype(int)

    churn_by_cluster = (
        churn_flag.groupby(df_result["cluster"])
        .agg(["count", "mean"])
        .rename(columns={"mean": "churn_rate"})
    )

    churn_by_cluster["churn_rate_pct"] = (
        churn_by_cluster["churn_rate"] * 100
    ).round(2)

    display(churn_by_cluster)
else:
    print(f"Kolom {TARGET!r} tidak ditemukan.")

In [ ]:
if TARGET in df_result.columns:
    # Catatan: nilai "Yes"/"No" diubah ke 1/0 agar bisa dirata-ratakan
    churn_plot = df_result.groupby("cluster")[TARGET].apply(lambda s: (s == "Yes").mean()) * 100

    churn_plot.plot(kind="bar", figsize=(7,4))
    plt.ylabel("Churn Rate (%)")
    plt.title("Churn Rate berdasarkan Cluster")
    plt.xticks(rotation=0)
    plt.grid(axis="y", alpha=0.3)
    plt.show()

# Bagian D — Interpretasi dan Rekomendasi

## 18. Insight Utama

1. **Insight 1: Risiko churn berbeda cukup jauh antar-segmen.**
   - Bukti dari data: churn berkisar sekitar 11% (cluster 0 dan 3) sampai sekitar 23% (cluster 2), dengan rata-rata keseluruhan 16,8%.
   - Makna: satu program retensi untuk semua pelanggan akan kurang tepat sasaran.

2. **Insight 2: Segmen bernilai tinggi justru yang paling berisiko.**
   - Bukti dari data: cluster 2 memiliki tagihan median tertinggi (sekitar 115) dan pemakaian tertinggi (sekitar 159 GB), dengan churn tertinggi (sekitar 23%).
   - Makna: churn di segmen ini lebih mahal bagi perusahaan karena setiap pelanggan yang hilang membawa pendapatan bulanan lebih besar.

3. **Insight 3: Lama berlangganan berkaitan dengan churn yang lebih rendah.**
   - Bukti dari data: cluster 3 (tenure median 47 bulan) churn sekitar 11,5%, sedangkan cluster 1 (tenure median 19 bulan) sekitar 20%; cluster 3 juga paling banyak berkontrak dua tahun.
   - Makna: periode awal berlangganan adalah masa rawan; sebaiknya ada perhatian khusus pada pelanggan baru.

## 19. Rekomendasi

| Cluster / Temuan | Rekomendasi | Alasan |
|---|---|---|
| Cluster 2: Pemakai Berat Tagihan Tinggi | Tawarkan paket berkapasitas lebih besar atau diskon loyalitas, serta pantau kualitas layanan untuk akun berpemakaian tinggi. | Churn tertinggi (sekitar 23%) dan nilai tagihan terbesar. |
| Cluster 1: Pelanggan Baru Menengah | Program onboarding dan check-in pada bulan-bulan awal; dorong peralihan ke kontrak 1-2 tahun. | Tenure paling pendek dan churn tinggi (sekitar 20%). |
| Cluster 3 dan 0 | Pertahankan dengan komunikasi rutin berbiaya rendah; untuk cluster 0 kaji paket alternatif atau edukasi layanan. | Churn terendah (sekitar 11%), tidak perlu insentif mahal. |

Rekomendasi ini perlu diuji terlebih dahulu (misalnya A/B test) sebelum diterapkan penuh.

## 20. Keterbatasan Analisis

- **Pemilihan fitur:** hanya 3 fitur yang dipakai. Segmen akan berubah jika fitur lain ikut masuk. Sel di bawah menambahkan `satisfaction_score` sebagai uji sensitivitas; silhouette turun (sekitar 0,23) dan profil cluster bergeser.
- **Scaling:** hasil bergantung pada `StandardScaler`; metode lain (misalnya `RobustScaler`) atau pembobotan fitur dapat mengubah cluster.
- **Asumsi bentuk cluster:** K-Means condong ke cluster kompak berbentuk bulat dengan ukuran serupa. Silhouette sekitar 0,32 menunjukkan struktur lemah sampai sedang, sehingga batas antar-segmen tidak tegas.
- **Outlier:** tidak ditemukan outlier ekstrem (|z| > 3), tetapi K-Means tetap sensitif terhadapnya pada data lain.
- **Stabilitas:** pada data ini hasil stabil terhadap seed (ARI ≥ 0,995), tetapi stabilitas terhadap sampel data atau pilihan fitur belum diuji menyeluruh.
- **Interpretasi churn:** hanya asosiasi deskriptif, bukan sebab-akibat, dan tanpa uji signifikansi.

In [ ]:
# Uji sensitivitas: tambahkan satisfaction_score sebagai fitur keempat
feat_alt = features + ["satisfaction_score"]
X_alt = StandardScaler().fit_transform(df[feat_alt])

rows = []
for k in range(2, 7):
    m = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_alt)
    rows.append((k, m.inertia_, silhouette_score(X_alt, m.labels_)))
pd.DataFrame(rows, columns=["k", "inertia", "silhouette"])

## 21. Kesimpulan

1. **Jumlah cluster:** k = 4, berdasarkan Elbow, Silhouette (puncak 0,322), interpretabilitas, ukuran cluster, dan stabilitas.
2. **Karakter utama:** Pemakai Ringan Hemat; Pelanggan Baru Menengah; Pemakai Berat Tagihan Tinggi; Loyal Jangka Panjang.
3. **Cluster paling penting:** Pemakai Berat Tagihan Tinggi, karena churn tertinggi (sekitar 23%) dengan nilai tagihan terbesar; berikutnya Pelanggan Baru Menengah (sekitar 20%).
4. **Rekomendasi utama:** retensi khusus untuk pemakai berat bertagihan tinggi, onboarding untuk pelanggan baru, dan komunikasi berbiaya rendah untuk segmen loyal. Struktur cluster lemah sampai sedang dan hubungan dengan churn bersifat asosiasi, jadi rekomendasi perlu diuji sebelum diterapkan penuh.

# Checklist Pengumpulan

- [ ] Identitas kelompok lengkap
- [ ] Pertanyaan analisis dijelaskan
- [ ] Memilih 3–5 fitur dan memberi alasan
- [ ] `churn_next_30d` tidak digunakan sebagai fitur clustering
- [ ] Preprocessing dijelaskan
- [ ] Scaling dilakukan
- [ ] Menguji `k = 2` sampai `6`
- [ ] Elbow/WCSS ditampilkan
- [ ] Silhouette Score ditampilkan
- [ ] Pemilihan `k` diberi argumentasi
- [ ] Profiling cluster dilakukan
- [ ] Cluster diberi nama/persona
- [ ] Minimal 2 visualisasi
- [ ] Churn rate dibandingkan setelah clustering
- [ ] Minimal 3 insight
- [ ] 2–3 rekomendasi
- [ ] Keterbatasan dibahas
- [ ] Semua cell dapat dijalankan dari awal sampai akhir
